# Preprocesamiento tabular — justificación y verificación

Este notebook **fundamenta con evidencia** las decisiones del preprocesamiento (`src/features/`) y
**verifica** la salida del entrypoint `python -m src.features.preprocess`:

1. patrón de valores faltantes → estrategia de imputación,
2. por qué se excluye `biopsed` (leakage),
3. ajuste del transformador **solo con train** (anti-leakage),
4. efecto del transformador (antes/después),
5. chequeos sobre `data/processed/tabular.parquet`.

Requiere haber corrido antes el split (`data/splits.csv`) y el preprocesamiento.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.data.dataset import load_with_splits
from src.data.dataset_utils import PROCESSED_DIR, TARGET, MALIGNANT
from src.features.tabular import build_tabular_transformer
from src.features.tabular_utils import NUMERIC, CATEGORICAL, select_features

df = load_with_splits()
X = select_features(df)
print("filas:", len(df), "| features de entrada:", X.shape[1])

## 1. Patrón de valores faltantes

Cuantificamos el faltante y mostramos que **no es aleatorio**: hay un bloque de datos del paciente
que falta en conjunto (faltante estructural), lo que motiva usar **mediana + indicador de faltante**
en numéricas y **"missing" como categoría** en categóricas.

In [ ]:
miss = (X.isna().mean() * 100).sort_values(ascending=False)
miss[miss > 0].round(1)

In [ ]:
# ¿faltan juntas? bloque de encuesta del paciente (age SI esta siempre presente)
block = ["gender", "fitspatrick", "diameter_1", "smoke", "drink"]
all_missing = X[block].isna().all(axis=1).sum()
any_missing = X[block].isna().any(axis=1).sum()
print(f"filas con TODO el bloque faltante: {all_missing}")
print(f"filas con ALGO del bloque faltante: {any_missing}")
print("-> si ambos números son parecidos, el bloque falta en conjunto (estructural)")

In [ ]:
# columna numerica con faltantes (elegida dinamicamente) para los ejemplos
REF = next((c for c in NUMERIC if X[c].isna().any()), None)
print("columna de referencia con faltantes:", REF)

# ¿el faltar se asocia con la clase? (senal de MNAR)
df["_missing_ref"] = X[REF].isna()
pd.crosstab(df[TARGET], df["_missing_ref"], normalize="columns").round(3)

Si la distribución de clases difiere entre filas con y sin dato, el faltante **depende de la clase**
(MNAR) → esconderlo con una imputación simple perdería señal. Por eso agregamos una **columna
indicadora** que le dice al modelo "este valor no estaba".

## 2. Por qué excluimos `biopsed` de las features

`biopsed` indica si la lesión fue biopsiada, y **la decisión de biopsiar depende de la sospecha
clínica**, que correlaciona con el diagnóstico. Usarlo como feature sería **leakage**: le daríamos al
modelo una pista del label que en la práctica no es un input neutro.

In [ ]:
df["malignant"] = df[TARGET].isin(MALIGNANT)
print("Tasa de biopsia por malignidad:")
print(df.groupby("malignant")["biopsed"].apply(lambda s: (s == "True").mean()).round(3))
print()
print("Biopsia por diagnóstico:")
pd.crosstab(df[TARGET], df["biopsed"], normalize="index").round(3)

## 3. El transformador se ajusta SOLO con train (anti-leakage)

Las medianas, categorías y escalas se aprenden **únicamente del conjunto de train**. Si se ajustaran
con todo el dataset, se filtraría información de val/test. Lo comprobamos comparando la mediana que
aprende el transformador con la mediana de train (deben coincidir) vs la de todo el dataset.

In [ ]:
train = df["split"] == "train"
transformer = build_tabular_transformer()
transformer.fit(X[train])

imp = transformer.named_transformers_["num"].named_steps["impute"]
pd.DataFrame({
    "mediana_transformer": imp.statistics_[:len(NUMERIC)],
    "mediana_train": X[train][NUMERIC].median().values,
    "mediana_dataset_completo": X[NUMERIC].median().values,
}, index=NUMERIC).round(2)

## 4. Efecto del transformador (antes / después)

Tomamos una fila con la columna de referencia faltante (elegida arriba) y mostramos que queda
imputada, con el indicador en 1 y sin NaN.

In [ ]:
Xt = transformer.transform(X)
names = transformer.get_feature_names_out()
Xt = pd.DataFrame(Xt, columns=names, index=X.index)

print("dimensiones: entrada", X.shape, "-> salida", Xt.shape)
print("¿quedan NaN?", bool(Xt.isna().any().any()))

idx = X.index[X[REF].isna()][0]
print(f"\nfila {idx} ({REF} faltante en crudo):")
print(X.loc[idx, [REF, "gender"]])
cols = [c for c in names if REF in c.lower()][:4]
print(f"\nmisma fila transformada (columnas de {REF}):")
print(Xt.loc[idx, cols])

In [ ]:
# expansion por one-hot y columnas indicadoras
print("columnas de entrada:", X.shape[1])
print("columnas de salida:", Xt.shape[1])
print("indicadores de faltante:", [c for c in names if "missingindicator" in c])
print("ejemplo one-hot region:", [c for c in names if "region" in c][:6])

## 5. Verificación de la salida del entrypoint

Cargamos `data/processed/tabular.parquet` (lo que generó `src.features.preprocess`) y chequeamos que
está bien formado.

In [ ]:
path = PROCESSED_DIR / "tabular.parquet"
proc = pd.read_parquet(path)
print("shape:", proc.shape)
print("split:", proc["split"].value_counts().to_dict())
feat_cols = [c for c in proc.columns if c not in ("img_id", "split", TARGET)]
print("features:", len(feat_cols))
print("¿NaN en features?", bool(proc[feat_cols].isna().any().any()))
assert set(proc["img_id"]) == set(df["img_id"]), "faltan filas en el parquet"
print("OK: una fila por imagen, sin NaN, con split y target")
proc.head(3)

## 6. Conclusiones

- **Faltante estructural y MNAR** → mediana + indicador (numéricas) y `"missing"` como categoría
  (categóricas): no inventamos datos y preservamos la señal de "dato ausente".
- **`biopsed` excluido**: correlaciona con el diagnóstico → sería leakage.
- **Ajuste solo con train**: las estadísticas del preprocesamiento no ven val/test.
- **Salida verificada**: `tabular.parquet` tiene una fila por imagen, sin NaN, con `split` y target,
  lista para el modelo. El transformador queda serializado para reusarlo idéntico en la API (anti-skew).